<a href="https://colab.research.google.com/github/jc13605-0721/ECE_6143_ML_Project/blob/main/HW1_ChipChat/chipchat_exampleB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Getting set up

## Setting up the notebook

In [ ]:
### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
iverilog is already the newest version (11.0-1.1).
0 upgraded, 0 newly installed, 0 to remove 

## Use LLM to generate Verilog code from natural language description

### Example 8: Traffic Light

In [ ]:
!mkdir -p traffic_light/
!cd traffic_light/ && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/traffic_light_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  2018  100  2018    0     0  12087      0 --:--:-- --:--:-- --:--:-- 12156


In [ ]:
verilog_generation_prompt = '''
I need help writing a Verilog module for a simple traffic light controller.

The design should be a synchronous finite state machine driven by a clock. It has an active-low reset.

Inputs:
- clk
- reset (active low)

Outputs:
- red
- yellow
- green

The behavior should be as follows:
- After reset, the traffic light starts in the red state.
- The light stays red for 32 clock cycles, then switches to green.
- The light stays green for 20 clock cycles, then switches to yellow.
- The light stays yellow for 10 clock cycles, then switches back to red.
- This sequence should repeat continuously.

Only one light should be asserted at any time.

A simple FSM with a state register and a counter is fine. Please make sure the reset behavior and timing are handled correctly.

The code should be synthesizable Verilog (no delays, no initial blocks).
Please output only the complete module code.

'''

In [ ]:
from openai import OpenAI
from google.colab import userdata
import os

# Load API key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Create OpenAI client
client = OpenAI()

completion = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": verilog_generation_prompt}],
    max_tokens=1024,
)

print(completion.choices[0].message.content)


```verilog
module traffic_light_controller (
    input wire clk,
    input wire reset_n,
    output reg red,
    output reg yellow,
    output reg green
);

    // State encoding
    typedef enum reg [1:0] {
        RED = 2'b00,
        GREEN = 2'b01,
        YELLOW = 2'b10
    } state_t;

    state_t current_state, next_state;
    reg [5:0] counter; // 6 bits can count up to 63

    // State register
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            current_state <= RED;
            counter <= 0;
        end else begin
            // State transition
            if (counter == get_state_duration(current_state)) begin
                current_state <= next_state;
                counter <= 0;
            end else begin
                counter <= counter + 1;
            end
        end
    end

    // Next state logic
    always @(*) begin
        case (current_state)
            RED: next_state = GREEN;
            GREEN: next_state = YELLOW;
  

In [ ]:
output_verilog_code = '''
module traffic_light_controller (
    input wire clk,
    input wire reset_n,
    output reg red,
    output reg yellow,
    output reg green
);

    // State encoding
    typedef enum reg [1:0] {
        RED = 2'b00,
        GREEN = 2'b01,
        YELLOW = 2'b10
    } state_t;

    state_t current_state, next_state;
    reg [5:0] counter; // 6 bits can count up to 63

    // State register
    always @(posedge clk or negedge reset_n) begin
        if (!reset_n) begin
            current_state <= RED;
            counter <= 0;
        end else begin
            // State transition
            if (counter == get_state_duration(current_state)) begin
                current_state <= next_state;
                counter <= 0;
            end else begin
                counter <= counter + 1;
            end
        end
    end

    // Next state logic
    always @(*) begin
        case (current_state)
            RED: next_state = GREEN;
            GREEN: next_state = YELLOW;
            YELLOW: next_state = RED;
            default: next_state = RED; // Fallback
        endcase
    end

    // Output logic
    always @(*) begin
        case (current_state)
            RED: begin
                red = 1;
                yellow = 0;
                green = 0;
            end
            GREEN: begin
                red = 0;
                yellow = 0;
                green = 1;
            end
            YELLOW: begin
                red = 0;
                yellow = 1;
                green = 0;
            end
            default: begin
                red = 0;
                yellow = 0;
                green = 0; // Default safe state
            end
        endcase
    end

    // Function to return the duration for each state
    function [5:0] get_state_duration(input state_t state);
        begin
            case (state)
                RED: get_state_duration = 32; // 32 clock cycles for red
                GREEN: get_state_duration = 20; // 20 clock cycles for green
                YELLOW: get_state_duration = 10; // 10 clock cycles for yellow
                default: get_state_duration = 0; // Default case
            endcase
        end
    endfunction

endmodule
'''
filename = "traffic_light/traffic_light.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [ ]:
!cd traffic_light/ && iverilog -g2012 -o traffic_light.vvp traffic_light.v traffic_light_tb.v && vvp traffic_light.vvp

Test case 1 passed - FSM transitioned to    RED after 0 clock cycles.
Test case 2 passed - FSM transitioned to  GREEN after 32 clock cycles.
Test case 3 passed - FSM transitioned to YELLOW after 20 clock cycles.
Test case 4 passed - FSM transitioned to    RED after 7 clock cycles.
All test cases passed!
